# EfficientNetV2-S — Progressive SSL with Test-Camera Seeding

## Strategy

1. **Seed training** — use only the ~60 images from train2 that come from test-camera views,
   plus a tiny random undersample (5%) of the remaining train2 cameras.
   This biases the initial model toward the test domain.

2. **Hold-out reserve** — the 95% of non-test-camera train2 images not used in step 1
   are kept in a shuffled reserve and released in equal chunks across 12 SSL iterations.

3. **Iterative progressive SSL** (12 × 2 epochs) — each iteration:
   - Re-generates pseudo-labels for all test instances with the current model
   - Filters to very high confidence (99 % → 95 %, decreasing schedule)
   - Adds the next reserve chunk (real labeled train2 data)
   - Fine-tunes for 2 epochs on the combined pool

4. **Final inference** — submission CSV produced from the last checkpoint.

Camera-specific augmentations from `EficientNet_Augmentacja.ipynb` are preserved throughout.


In [ ]:
# ── Installation ─────────────────────────────────────────────────────────────
!pip install albumentations -q

# ── Kaggle credentials ───────────────────────────────────────────────────────
from google.colab import files
files.upload()   # upload kaggle.json

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# ── Download competition data ─────────────────────────────────────────────────
!kaggle competitions download -c multi-view-pig-posture-recognition
!unzip -q multi-view-pig-posture-recognition.zip
!ls

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, ast, copy, re, time
from collections import Counter

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ── GPU ───────────────────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    _ = torch.zeros(1).cuda(); torch.cuda.synchronize()
    x = torch.randn(10000, 10000, device='cuda')
    t0 = time.time(); _ = x @ x; torch.cuda.synchronize()
    print(f'GPU test: {time.time()-t0:.3f}s')
    del x; torch.cuda.empty_cache()

In [ ]:
# ── Class labels ──────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0: 'Lateral_lying_left',
    1: 'Lateral_lying_right',
    2: 'Sitting',
    3: 'Standing',
    4: 'Sternal_lying',
}
NUM_CLASSES = len(CLASS_NAMES)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('multiview_pig_posture_recognition')
TRAIN2_IMGS = BASE_DIR / 'train2_images'
TEST_IMGS   = BASE_DIR / 'test_images'

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED               = 42
BATCH_SIZE         = 32

# Seed training (on the small test-camera-biased initial set)
UNDERSAMPLE_RATIO  = 0.05    # fraction of non-test-camera train2 images to keep initially
EPOCHS_INITIAL     = 15      # epochs for the seed training run
LR_INITIAL         = 1e-4
WD_INITIAL         = 1e-4    # weight decay to regularise the small initial dataset

# Progressive SSL
N_ITERATIONS       = 12      # number of SSL fine-tuning iterations (>= 10)
EPOCHS_PER_ITER    = 2       # epochs per iteration
LR_SSL             = 3e-5    # constant LR during SSL fine-tuning

# Confidence threshold schedule: starts very strict, relaxes linearly to 0.95
SSL_THRESHOLDS     = np.linspace(0.99, 0.95, N_ITERATIONS)

# Save paths
SAVE_INITIAL       = 'effnetv2_initial.pt'
SAVE_SSL           = 'effnetv2_ssl_progressive.pt'
SUBMISSION_PATH    = 'submission_ssl_progressive.csv'

np.random.seed(SEED)
torch.manual_seed(SEED)

print('Thresholds per iteration:')
for i, t in enumerate(SSL_THRESHOLDS):
    print(f'  Iter {i+1:>2}: {t:.4f}')

In [ ]:
# ── Camera metadata helpers ───────────────────────────────────────────────────
def parse_camera_meta(image_id):
    m = re.match(r'(pen\d+)_(orb|tur)_(cam\d+)_', str(image_id))
    if m:
        return m.group(1), m.group(2), m.group(3)
    return 'unknown', 'unknown', 'unknown'

def add_camera_cols(df):
    df['pen']      = df['image_id'].apply(lambda x: parse_camera_meta(x)[0])
    df['cam_type'] = df['image_id'].apply(lambda x: parse_camera_meta(x)[1])
    df['cam_num']  = df['image_id'].apply(lambda x: parse_camera_meta(x)[2])
    df['camera']   = df['pen'] + '_' + df['cam_type'] + '_' + df['cam_num']
    return df

# ── Load CSVs ─────────────────────────────────────────────────────────────────
train2 = pd.read_csv(BASE_DIR / 'train2.csv')
train2['source']      = 'train2'
train2['bbox_parsed'] = train2['bbox'].apply(ast.literal_eval)
train2['class_name']  = train2['class_id'].map(CLASS_NAMES)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / 'test.csv')
test['source']      = 'test'
test['bbox_parsed'] = test['bbox'].apply(ast.literal_eval)
test = add_camera_cols(test)

print(f'Train2: {len(train2):,} instances  |  {train2["image_id"].nunique():,} images')
print(f'Test:   {len(test):,} instances   |  {test["image_id"].nunique():,} images')

## Train2 / Test Camera Analysis & Data Split

Identify which cameras appear in **both** train2 and test — these are the ~60 test-camera images  
included in Training Set 2. They form the seed of our initial training set.

In [ ]:
# ── Identify shared cameras ───────────────────────────────────────────────────
test_cams   = set(test['camera'].unique())
train2_cams = set(train2['camera'].unique())
shared_cams = test_cams & train2_cams

print(f'Test cameras ({len(test_cams)}):    {sorted(test_cams)}')
print(f'Train2 cameras ({len(train2_cams)}):  {sorted(train2_cams)}')
print(f'Shared cameras ({len(shared_cams)}): {sorted(shared_cams)}')

# ── Split train2 into test-camera vs. other ────────────────────────────────────
test_cam_df = train2[train2['camera'].isin(shared_cams)].copy()
other_df    = train2[~train2['camera'].isin(shared_cams)].copy()

print(f'\nTest-camera instances in train2: {len(test_cam_df):,} '
      f'from {test_cam_df["image_id"].nunique():,} images')
print(f'Other-camera instances in train2: {len(other_df):,} '
      f'from {other_df["image_id"].nunique():,} images')

# ── Undersample other cameras (image-level to avoid leakage within images) ────
other_image_ids = other_df['image_id'].unique()
n_initial_imgs  = max(1, int(len(other_image_ids) * UNDERSAMPLE_RATIO))
rng             = np.random.default_rng(SEED)
initial_img_ids = rng.choice(other_image_ids, n_initial_imgs, replace=False)
initial_other   = other_df[other_df['image_id'].isin(initial_img_ids)].copy()
held_out_df     = other_df[~other_df['image_id'].isin(initial_img_ids)].copy()

# Shuffle held-out images for progressive release
held_out_imgs   = held_out_df['image_id'].unique().copy()
rng.shuffle(held_out_imgs)
chunk_size      = max(1, len(held_out_imgs) // N_ITERATIONS)

# ── Initial training set ───────────────────────────────────────────────────────
initial_train_df = pd.concat([test_cam_df, initial_other], ignore_index=True)

print(f'\n--- Initial training set ---')
print(f'  Test-camera instances:          {len(test_cam_df):,}')
print(f'  Undersampled other instances:   {len(initial_other):,} '
      f'(~{100*UNDERSAMPLE_RATIO:.0f}% of {len(other_image_ids):,} other images)')
print(f'  TOTAL initial instances:        {len(initial_train_df):,}')
print(f'\n--- Held-out reserve ---')
print(f'  Held-out images:                {len(held_out_imgs):,}')
print(f'  Held-out instances:             {len(held_out_df):,}')
print(f'  Chunk size per iteration:       ~{chunk_size} images / '
      f'~{int(len(held_out_df)/N_ITERATIONS)} instances')

print(f'\nClass distribution in initial set:')
for cname, cnt in initial_train_df['class_name'].value_counts().items():
    print(f'  {cname:25s}: {cnt:,}')

In [ ]:
# ── Image loading & crop utilities ────────────────────────────────────────────
def load_image(image_id, source):
    folder = {'train2': TRAIN2_IMGS, 'test': TEST_IMGS}[source]
    return Image.open(folder / image_id).convert('RGB')


def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    """Crop pig instance from image using bounding box [x, y, w, h] with padding."""
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = x - w * padding;      y1 = y - h * padding
    x2 = x + w + w * padding;  y2 = y + h + h * padding
    if make_square:
        side   = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2 = cx - side / 2, cx + side / 2
        y1, y2 = cy - side / 2, cy + side / 2
    x1 = max(0, int(round(x1)));      y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)));  y2 = min(img_h, int(round(y2)))
    return image.crop((max(x1, 0), max(y1, 0), max(x2, x1 + 1), max(y2, y1 + 1)))

In [ ]:
# ── Camera-specific augmentation functions (from EficientNet_Augmentacja.ipynb) ──
#
# These transform images from train cameras to simulate the visual appearance
# of their corresponding test-camera counterparts (brightness, colour, mirror).
# Cameras that include a horizontal flip require label remapping:
#   Lateral_lying_left (0) <-> Lateral_lying_right (1)

FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}


def aug_pen2_tur_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    img = TF.adjust_brightness(img, brightness_factor=0.95)
    return img


def aug_pen1_tur_cam2(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    img = TF.adjust_saturation(img, saturation_factor=0.9)
    return img


def aug_pen2_orb_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=0.6)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    img_np = np.array(img).astype(float)
    img_np[:, :, 0] = (img_np[:, :, 0] * 0.85).clip(0, 255)
    img_np[:, :, 1] = (img_np[:, :, 1] * 1.10).clip(0, 255)
    img_np[:, :, 2] = (img_np[:, :, 2] * 0.80).clip(0, 255)
    img_shifted = Image.fromarray(img_np.clip(0, 255).astype(np.uint8))
    img_grey    = TF.to_grayscale(img_shifted, num_output_channels=3)
    grey_np     = np.array(img_grey).astype(float)
    blended     = (0.65 * img_np + 0.35 * grey_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)


def aug_pen2_tur_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    img = TF.adjust_saturation(img, saturation_factor=0.85)
    return img


def aug_pen2_orb_cam2(img):
    img = TF.adjust_brightness(img, brightness_factor=0.60)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    img_grey = TF.to_grayscale(img, num_output_channels=3)
    img_np   = np.array(img).astype(float)
    grey_np  = np.array(img_grey).astype(float)
    blended  = (0.65 * img_np + 0.35 * grey_np).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)


# pen1_orb_cam1, pen1_orb_cam2 — no test-camera counterpart, no augmentation
CAMERA_AUG_FN = {
    'pen2_tur_cam1': (aug_pen2_tur_cam1, True),
    'pen1_tur_cam2': (aug_pen1_tur_cam2, True),
    'pen2_orb_cam1': (aug_pen2_orb_cam1, True),
    'pen2_tur_cam2': (aug_pen2_tur_cam2, False),
    'pen2_orb_cam2': (aug_pen2_orb_cam2, False),
}

print('Camera augmentations:')
for cam, (_, flip) in CAMERA_AUG_FN.items():
    print(f'  {cam:20s}  flip_label={flip}')

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15):
        self.std, self.p = std, p
    def __call__(self, tensor):
        if torch.rand(1).item() < self.p:
            tensor = torch.clamp(tensor + torch.randn_like(tensor) * self.std, 0., 1.)
        return tensor


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Used after camera-specific PIL augmentation (supervised training)
BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Clean transform for test inference and pseudo-label confidence scoring
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# For pseudo-labeled test crops during SSL fine-tuning.
# General augmentation only — test images are already in their native camera domain.
SSL_TRANSFORM = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined ✓')

In [ ]:
# ── Dataset classes ───────────────────────────────────────────────────────────

class PigDatasetCameraAug(Dataset):
    """
    Labeled training dataset with optional camera-specific augmentation.

    For every row the original crop is included.
    If the camera key appears in camera_aug_fn, an augmented copy is also added.
    For cameras whose augmentation includes a horizontal flip the label is
    remapped via FLIP_LABEL_MAP (Lateral_lying_left <-> Lateral_lying_right).
    """
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.camera_aug_fn  = camera_aug_fn
        self.base_transform = base_transform
        df = df.reset_index(drop=True)
        self.df = df

        self.samples = []  # (iloc_idx, do_aug, label)
        for idx, row in df.iterrows():
            cam   = row.get('camera', 'unknown')
            label = int(row['class_id'])
            self.samples.append((idx, False, label))
            if is_train and cam in camera_aug_fn:
                _, flip = camera_aug_fn[cam]
                aug_label = FLIP_LABEL_MAP[label] if flip else label
                self.samples.append((idx, True, aug_label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        row_idx, do_aug, label = self.samples[idx]
        row  = self.df.iloc[row_idx]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'], padding=0.12, make_square=True)
        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row['camera']]
            crop = aug_fn(crop)
        return self.base_transform(crop), label


class PigTestDataset(Dataset):
    """Returns (image_tensor, row_id_str) — used for inference and pseudo-labeling."""
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'], padding=0.12, make_square=True)
        return self.transform(crop), str(row['row_id'])


class PseudoLabelDataset(Dataset):
    """
    High-confidence pseudo-labeled test instances for SSL fine-tuning.
    Requires columns: image_id, source, bbox_parsed, class_id.
    Uses SSL_TRANSFORM (general augmentation, no camera-specific transforms).
    """
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'], padding=0.12, make_square=True)
        return self.transform(crop), int(row['class_id'])


print('Dataset classes defined ✓')

In [ ]:
# ── WeightedRandomSampler loader builder ──────────────────────────────────────
def build_weighted_loader(dataset, labels, batch_size, num_workers=2):
    """Build a DataLoader with class-balanced WeightedRandomSampler."""
    labels = np.asarray(labels, dtype=int)
    class_counts  = np.bincount(labels, minlength=NUM_CLASSES)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_w      = class_weights[labels]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_w),
        num_samples=len(sample_w),
        replacement=True,
    )
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=num_workers, pin_memory=True)


# ── Build the initial supervised loader ───────────────────────────────────────
init_sup_ds   = PigDatasetCameraAug(initial_train_df, CAMERA_AUG_FN, BASE_TRANSFORM)
init_labels   = [s[2] for s in init_sup_ds.samples]
init_loader   = build_weighted_loader(init_sup_ds, init_labels, BATCH_SIZE)

init_counts = np.bincount(init_labels, minlength=NUM_CLASSES)
print(f'Initial dataset — {len(initial_train_df):,} instances '
      f'→ {len(init_sup_ds):,} samples after camera aug  '
      f'| {len(init_loader)} batches')
print('Class distribution after camera aug:')
for cid, cnt in enumerate(init_counts):
    print(f'  {CLASS_NAMES[cid]:25s}: {cnt:,}')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
def create_model():
    m = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m.to(DEVICE)


model = create_model()
total_params = sum(p.numel() for p in model.parameters())
print(f'EfficientNetV2-S  |  {total_params:,} parameters')

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_preds, all_targets = 0.0, [], []
    n = len(loader)
    for bi, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        all_preds.extend(out.argmax(1).detach().cpu().tolist())
        all_targets.extend(labels.cpu().tolist())
        bar = '█' * int(30*(bi+1)/n) + '░' * (30 - int(30*(bi+1)/n))
        print(f'\r  |{bar}| {100*(bi+1)/n:5.1f}%  loss={loss.item():.4f}',
              end='', flush=True)
    print()
    return (total_loss / len(loader.dataset),
            accuracy_score(all_targets, all_preds),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def generate_pseudo_labels(model, test_df, transform):
    """
    Run inference on all test instances.
    Returns a copy of test_df with added columns 'class_id' and 'confidence'.
    Confidence = max softmax probability across classes.
    """
    loader = DataLoader(
        PigTestDataset(test_df, transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
    )
    all_preds, all_confs = [], []
    model.eval()
    with torch.no_grad():
        for bi, (imgs, _) in enumerate(loader):
            probs      = F.softmax(model(imgs.to(DEVICE)), dim=1)
            confs, preds = probs.max(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_confs.extend(confs.cpu().tolist())
            bar = '█' * int(20*(bi+1)/len(loader)) + '░' * (20 - int(20*(bi+1)/len(loader)))
            print(f'\r  [pseudo] |{bar}| {bi+1}/{len(loader)}', end='', flush=True)
    print()
    result = test_df.copy().reset_index(drop=True)
    result['class_id']   = all_preds
    result['confidence'] = all_confs
    return result


print('Utilities defined ✓')

## Seed Training

Train on the small initial set (test-camera images + 5 % of other cameras).  
Weight decay regularises the model on this tiny dataset.

In [ ]:
print('=' * 65)
print('SEED TRAINING')
print(f'  Dataset: {len(init_sup_ds):,} samples  |  '
      f'{EPOCHS_INITIAL} epochs  |  lr={LR_INITIAL}  |  wd={WD_INITIAL}')
print('=' * 65)

criterion_init = nn.CrossEntropyLoss()
optimizer_init = torch.optim.Adam(model.parameters(),
                                  lr=LR_INITIAL, weight_decay=WD_INITIAL)
scheduler_init = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_init, T_max=EPOCHS_INITIAL, eta_min=1e-6
)

best_f1_init, best_state_init, history_init = 0.0, None, []

for epoch in range(EPOCHS_INITIAL):
    if epoch % 5 == 0 and DEVICE.type == 'cuda':
        _x = torch.randn(10000, 10000, device=DEVICE)
        _t = time.time(); _ = _x @ _x; torch.cuda.synchronize()
        print(f'  [GPU] {time.time()-_t:.3f}s')
        del _x; torch.cuda.empty_cache()

    t0 = time.time()
    loss, acc, f1 = train_one_epoch(model, init_loader, optimizer_init, criterion_init)
    scheduler_init.step()
    elapsed = time.time() - t0

    history_init.append({'epoch': epoch+1, 'train_loss': loss,
                         'train_acc': acc, 'train_f1': f1})
    flag = '  ← best' if f1 > best_f1_init else ''
    print(f'[Seed {epoch+1:>2}/{EPOCHS_INITIAL}] ({elapsed/60:.1f}m)  '
          f'loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}  '
          f'lr={optimizer_init.param_groups[0]["lr"]:.1e}{flag}', flush=True)

    if f1 > best_f1_init:
        best_f1_init    = f1
        best_state_init = copy.deepcopy(model.state_dict())
        torch.save({'model_state_dict': best_state_init,
                    'class_names': CLASS_NAMES,
                    'best_f1': best_f1_init,
                    'epoch': epoch+1,
                    'phase': 'seed'}, SAVE_INITIAL)
        print(f'  Saved → {SAVE_INITIAL}', flush=True)

# Load best seed weights
model.load_state_dict(torch.load(SAVE_INITIAL, map_location=DEVICE)['model_state_dict'])
print(f'\nSeed training done.  Best F1: {best_f1_init:.4f}')

In [ ]:
# ── Seed training curve ───────────────────────────────────────────────────────
hist_df = pd.DataFrame(history_init)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], marker='o', markersize=4)
axes[0].set_title('Seed Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)
axes[1].plot(hist_df['epoch'], hist_df['train_f1'], marker='o', color='green', markersize=4)
axes[1].axhline(best_f1_init, color='red', linestyle='--', label=f'best={best_f1_init:.4f}')
axes[1].set_title('Seed Training Macro-F1'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Seed Training — test-camera biased initial set', fontweight='bold')
plt.tight_layout()
plt.savefig('seed_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## Progressive SSL — 12 Iterations × 2 Epochs

Each iteration:
1. **Pseudo-label** the full test set with the current model (fresh each round)
2. **Filter** to `confidence ≥ threshold` (schedule 0.99 → 0.95)
3. **Release** the next chunk of held-out labeled train2 images
4. **Fine-tune** 2 epochs on `supervised pool (with camera aug) + pseudo-labeled test (SSL aug)`
5. **Save** checkpoint


In [ ]:
# ── Progressive SSL loop ──────────────────────────────────────────────────────
accumulated_extra   = pd.DataFrame()   # held-out train2 released so far
iter_logs           = []               # per-iteration metrics
ssl_history_all     = []               # flat list of every epoch record
global_epoch        = EPOCHS_INITIAL   # offset for global epoch axis in final plot

criterion_ssl = nn.CrossEntropyLoss()

for it in range(N_ITERATIONS):
    threshold = float(SSL_THRESHOLDS[it])

    print(f'\n{"="*70}')
    print(f'ITERATION {it+1:>2}/{N_ITERATIONS}   threshold={threshold:.4f}   '
          f'global_epoch_offset={global_epoch}')
    print(f'{"="*70}')

    # ── 1. Generate pseudo-labels ─────────────────────────────────────────────
    pseudo_all = generate_pseudo_labels(model, test, VAL_TRANSFORM)
    pseudo_high = pseudo_all[pseudo_all['confidence'] >= threshold].copy()

    n_pseudo = len(pseudo_high)
    print(f'  Pseudo-labels above {threshold:.4f}: {n_pseudo:,} / {len(pseudo_all):,} '
          f'({100*n_pseudo/len(pseudo_all):.1f}%)')

    if n_pseudo > 0:
        dist = Counter(pseudo_high['class_id'].tolist())
        for cid, cnt in sorted(dist.items()):
            print(f'    [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}')

    # ── 2. Release next held-out chunk ────────────────────────────────────────
    c_start = it * chunk_size
    c_end   = min((it + 1) * chunk_size, len(held_out_imgs))
    chunk_img_ids = held_out_imgs[c_start:c_end]
    new_chunk     = held_out_df[held_out_df['image_id'].isin(chunk_img_ids)].copy()
    accumulated_extra = pd.concat([accumulated_extra, new_chunk], ignore_index=True)
    current_labeled   = pd.concat([initial_train_df, accumulated_extra], ignore_index=True)

    print(f'  Released {len(new_chunk):,} instances ({len(chunk_img_ids)} images) '
          f'→ labeled pool now {len(current_labeled):,} instances')

    # ── 3. Build combined dataset ─────────────────────────────────────────────
    sup_ds     = PigDatasetCameraAug(current_labeled, CAMERA_AUG_FN, BASE_TRANSFORM)
    sup_labels = [s[2] for s in sup_ds.samples]

    if n_pseudo > 0:
        pseudo_ds     = PseudoLabelDataset(pseudo_high, SSL_TRANSFORM)
        pseudo_labels = pseudo_high['class_id'].astype(int).tolist()
        combined_ds   = ConcatDataset([sup_ds, pseudo_ds])
        all_labels    = sup_labels + pseudo_labels
    else:
        print('  ⚠ No pseudo-labels above threshold — training on labeled data only')
        combined_ds = sup_ds
        all_labels  = sup_labels

    combined_loader = build_weighted_loader(combined_ds, all_labels, BATCH_SIZE)
    print(f'  Combined: {len(combined_ds):,} samples  |  {len(combined_loader)} batches')

    # ── 4. Fine-tune 2 epochs ─────────────────────────────────────────────────
    optimizer_ssl = torch.optim.Adam(model.parameters(), lr=LR_SSL)

    epoch_f1s, epoch_losses = [], []
    for ep in range(EPOCHS_PER_ITER):
        global_epoch += 1
        t0 = time.time()
        loss, acc, f1 = train_one_epoch(model, combined_loader, optimizer_ssl, criterion_ssl)
        elapsed = time.time() - t0
        epoch_f1s.append(f1)
        epoch_losses.append(loss)
        ssl_history_all.append({'global_epoch': global_epoch, 'iteration': it+1,
                                'train_loss': loss, 'train_f1': f1})
        print(f'  [Iter {it+1} Ep {ep+1}/{EPOCHS_PER_ITER}] ({elapsed/60:.1f}m)  '
              f'loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}', flush=True)

    iter_logs.append({
        'iteration':  it + 1,
        'threshold':  threshold,
        'n_pseudo':   n_pseudo,
        'n_labeled':  len(current_labeled),
        'n_combined': len(combined_ds),
        'final_f1':   epoch_f1s[-1],
        'final_loss': epoch_losses[-1],
    })

    # ── 5. Save checkpoint ────────────────────────────────────────────────────
    torch.save({
        'model_state_dict': model.state_dict(),
        'class_names':      CLASS_NAMES,
        'iteration':        it + 1,
        'threshold':        threshold,
        'final_f1':         epoch_f1s[-1],
        'n_pseudo':         n_pseudo,
    }, SAVE_SSL)
    print(f'  ✓ Saved → {SAVE_SSL}', flush=True)

print(f'\n{"="*70}')
print('ALL ITERATIONS COMPLETE')
print(f'{"="*70}')

In [ ]:
# ── SSL summary table ─────────────────────────────────────────────────────────
logs_df = pd.DataFrame(iter_logs)
print('Progressive SSL — iteration summary')
print(logs_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}'))

In [ ]:
# ── Full training history plot ────────────────────────────────────────────────
ssl_hist_df = pd.DataFrame(ssl_history_all)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Top-left: F1 over global epochs (seed + all SSL)
seed_epochs = list(range(1, EPOCHS_INITIAL + 1))
seed_f1s    = [r['train_f1'] for r in history_init]
ax = axes[0, 0]
ax.plot(seed_epochs, seed_f1s, marker='o', markersize=3,
        color='royalblue', label='Seed training')
ax.plot(ssl_hist_df['global_epoch'], ssl_hist_df['train_f1'],
        marker='s', markersize=3, color='darkorange', label='SSL iterations')
ax.axvline(EPOCHS_INITIAL, color='gray', linestyle=':', alpha=0.7, label='SSL start')
ax.set_title('Train Macro-F1 (all epochs)')
ax.set_xlabel('Global Epoch'); ax.grid(alpha=0.3); ax.legend()

# Top-right: Loss over global epochs
seed_losses = [r['train_loss'] for r in history_init]
ax = axes[0, 1]
ax.plot(seed_epochs, seed_losses, marker='o', markersize=3, color='royalblue', label='Seed')
ax.plot(ssl_hist_df['global_epoch'], ssl_hist_df['train_loss'],
        marker='s', markersize=3, color='darkorange', label='SSL')
ax.axvline(EPOCHS_INITIAL, color='gray', linestyle=':', alpha=0.7)
ax.set_title('Train Loss (all epochs)')
ax.set_xlabel('Global Epoch'); ax.grid(alpha=0.3); ax.legend()

# Bottom-left: F1 per SSL iteration (final epoch of each iter)
ax = axes[1, 0]
ax.bar(logs_df['iteration'], logs_df['final_f1'], color='steelblue', edgecolor='black')
ax.plot(logs_df['iteration'], logs_df['final_f1'],
        marker='o', color='red', linewidth=1.5, markersize=5)
ax.set_title('Final F1 per SSL Iteration')
ax.set_xlabel('Iteration'); ax.grid(alpha=0.3, axis='y')
ax2 = ax.twinx()
ax2.plot(logs_df['iteration'], logs_df['threshold'],
         marker='^', color='purple', linestyle='--', linewidth=1.5, label='Threshold')
ax2.set_ylabel('Confidence Threshold', color='purple')
ax2.legend(loc='upper right')

# Bottom-right: pseudo-label count and labeled pool size per iteration
ax = axes[1, 1]
ax.bar(logs_df['iteration'], logs_df['n_pseudo'],
       color='coral', edgecolor='black', label='Pseudo-labels')
ax.set_title('Pseudo-labels & Labeled Pool per Iteration')
ax.set_xlabel('Iteration'); ax.set_ylabel('Pseudo-label count', color='coral')
ax.grid(alpha=0.3, axis='y')
ax3 = ax.twinx()
ax3.plot(logs_df['iteration'], logs_df['n_labeled'],
         marker='D', color='navy', linewidth=2, label='Labeled pool size')
ax3.set_ylabel('Labeled instance count', color='navy')
ax3.legend(loc='upper left')

plt.suptitle('Progressive SSL Training Summary — EfficientNetV2-S',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('progressive_ssl_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Seed best F1:       {best_f1_init:.4f}')
print(f'SSL final iter F1:  {logs_df["final_f1"].iloc[-1]:.4f}')
print(f'SSL best iter F1:   {logs_df["final_f1"].max():.4f}  '
      f'(iter {logs_df["final_f1"].idxmax()+1})')

## Final Inference & Submission

In [ ]:
print('=' * 65)
print('FINAL INFERENCE')
print('=' * 65)

# Load the most recent SSL checkpoint
ckpt = torch.load(SAVE_SSL, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded checkpoint  iter={ckpt["iteration"]}  '
      f'f1={ckpt["final_f1"]:.4f}  threshold={ckpt["threshold"]:.4f}')

test_loader = DataLoader(
    PigTestDataset(test, VAL_TRANSFORM),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

all_row_ids, all_preds = [], []
with torch.no_grad():
    for bi, (imgs, row_ids) in enumerate(test_loader):
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().tolist()
        all_row_ids.extend(list(row_ids))
        all_preds.extend(preds)
        bar = '█' * int(30*(bi+1)/len(test_loader)) + '░' * (30-int(30*(bi+1)/len(test_loader)))
        print(f'\r  |{bar}| {bi+1}/{len(test_loader)}', end='', flush=True)

print(f'\n{len(all_preds):,} predictions generated')

# ── Build and validate submission ─────────────────────────────────────────────
submission = pd.DataFrame({'row_id': all_row_ids, 'class_id': all_preds})
sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')

assert set(submission['row_id']) == set(sample_sub['row_id']), \
    'row_id mismatch with sample_submission!'
assert submission['class_id'].between(0, 4).all(), \
    'class_id values out of range [0, 4]!'

submission = (
    submission.set_index('row_id')
              .reindex(sample_sub['row_id'])
              .reset_index()
)

print('\nPrediction distribution:')
for cid, cnt in sorted(submission['class_id'].value_counts().items()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}  ({100*cnt/len(submission):.1f}%)')

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSaved → {SUBMISSION_PATH}')
print(submission.head(10).to_string())

from google.colab import files
files.download(SAVE_SSL)
files.download(SUBMISSION_PATH)
print('\nModel + submission downloaded ✓')